# Harris County, TX — FEMA NFHL flood hazards (Phase 1, step 2)

Second link in the local chain. Pull FEMA NFHL flood hazard zones (`S_FLD_HAZ_AR`)
and base flood elevations (`S_BFE`) for the Harris County bbox via the public
ArcGIS REST endpoint, write to local parquet, and overlay on a folium map.

**Why REST and not the state GDB.** The CLAUDE.md canonical path is downloading
the Texas state geodatabase (~1.5 GB) and converting to GeoParquet. For a
single-county prototype that is wasteful. The REST service supports bbox queries
with paging and returns GeoJSON we can stream into geopandas. Cloud Phase 2 will
switch to the state GDB so we have a single bulk download per state.

**Inputs:** FEMA NFHL public MapServer (anonymous read).

**Outputs:**
- `data/raw/harris_nfhl_zones.parquet` — `S_FLD_HAZ_AR` polygons
- `data/raw/harris_nfhl_bfe.parquet` — `S_BFE` lines


In [ ]:
from pathlib import Path
import json
import time

import httpx
import geopandas as gpd
import pandas as pd
import folium


In [ ]:
# Harris County bbox — must match the buildings notebook so the joins line up.
MIN_LON, MAX_LON = -95.91, -94.90
MIN_LAT, MAX_LAT = 29.49, 30.18

NFHL_ROOT = "https://hazards.fema.gov/arcgis/rest/services/public/NFHL/MapServer"

REPO_ROOT = Path.cwd().resolve().parents[1]
RAW_DIR = REPO_ROOT / "data" / "raw"
ZONES_PATH = RAW_DIR / "harris_nfhl_zones.parquet"
BFE_PATH = RAW_DIR / "harris_nfhl_bfe.parquet"
RAW_DIR.mkdir(parents=True, exist_ok=True)
ZONES_PATH, BFE_PATH


## Discover layer IDs

NFHL MapServer layer numbering has shifted across releases (`S_FLD_HAZ_AR` has
lived at 27 and 28 at various points). Query the service root and resolve by
layer name so the notebook keeps working when FEMA reshuffles.


In [ ]:
client = httpx.Client(timeout=60.0)
meta = client.get(NFHL_ROOT, params={"f": "json"}).json()
layers_by_name = {layer["name"]: layer["id"] for layer in meta["layers"]}
ZONE_LAYER_ID = layers_by_name["Flood Hazard Zones"]
BFE_LAYER_ID = layers_by_name["Base Flood Elevations"]
ZONE_LAYER_ID, BFE_LAYER_ID


## Bbox fetch by OBJECTID batching

The REST service caps `maxRecordCount` at 1000–2000 features and falls over
on deep `resultOffset` paging (HTTP 500 past ~18k for Harris zones — empirical).
The reliable pattern, used here, is two-phase:

1. `returnIdsOnly=true` to grab every OBJECTID intersecting the bbox in one cheap call.
2. Batch the OIDs into chunks of 500 and request each batch with `objectIds=...`.

This sidesteps offset paging entirely and lets us retry individual batches
without risking a desync. `outSR=4326` keeps the server-side reprojection so we
don't have to deal with the Texas state-plane CRS.


In [ ]:
BATCH = 200  # POST'd in form body, kept moderate so a single bad batch is cheap to retry.

def _request_with_retry(url: str, params: dict) -> dict:
    for attempt in range(5):
        try:
            # POST so OBJECTID lists don't blow the GET URL length limit.
            resp = client.post(url, data=params)
            resp.raise_for_status()
            payload = resp.json()
            if isinstance(payload, dict) and payload.get("error"):
                raise httpx.HTTPError(f"esri error: {payload['error']}")
            return payload
        except (httpx.HTTPError, json.JSONDecodeError):
            if attempt == 4:
                raise
            time.sleep(2 ** attempt)
    raise RuntimeError("unreachable")


def fetch_layer(layer_id: int, out_fields: str, where: str = "1=1") -> gpd.GeoDataFrame:
    url = f"{NFHL_ROOT}/{layer_id}/query"
    bbox_params = {
        "where": where,
        "geometry": f"{MIN_LON},{MIN_LAT},{MAX_LON},{MAX_LAT}",
        "geometryType": "esriGeometryEnvelope",
        "inSR": "4326",
        "spatialRel": "esriSpatialRelIntersects",
        "f": "json",
    }
    ids_payload = _request_with_retry(url, {**bbox_params, "returnIdsOnly": "true"})
    oids = ids_payload.get("objectIds") or []
    print(f"  layer {layer_id}: {len(oids):,} object IDs in bbox")
    if not oids:
        return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")

    n_batches = (len(oids) - 1) // BATCH + 1
    frames: list[gpd.GeoDataFrame] = []
    for i in range(0, len(oids), BATCH):
        chunk = oids[i : i + BATCH]
        params = {
            "objectIds": ",".join(str(o) for o in chunk),
            "outFields": out_fields,
            "outSR": "4326",
            "f": "geojson",
        }
        payload = _request_with_retry(url, params)
        feats = payload.get("features", [])
        if feats:
            frames.append(gpd.GeoDataFrame.from_features(feats, crs="EPSG:4326"))
        print(f"    batch {i // BATCH + 1}/{n_batches}: {len(feats)} features")
    return gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), crs="EPSG:4326")


## Pull flood hazard zones

Skipping if the local parquet exists; delete the file to refetch.


In [ ]:
ZONE_FIELDS = "FLD_ZONE,ZONE_SUBTY,STATIC_BFE,V_DATUM,SFHA_TF,DEPTH,DFIRM_ID"

if ZONES_PATH.exists():
    print(f"already have {ZONES_PATH}; skipping fetch")
    zones = gpd.read_parquet(ZONES_PATH)
else:
    print("fetching S_FLD_HAZ_AR...")
    zones = fetch_layer(ZONE_LAYER_ID, out_fields=ZONE_FIELDS)
    zones.to_parquet(ZONES_PATH, compression="zstd")
    print(f"wrote {ZONES_PATH} ({len(zones):,} features)")
zones.head(3)


In [ ]:
zones.groupby("FLD_ZONE", dropna=False).size().sort_values(ascending=False).rename("n").to_frame()


## Pull base flood elevations

`S_BFE` is a line layer carrying the 1% annual-chance water-surface elevation
(`ELEV`, in feet, NAVD88 in most counties). Building-level depth is computed
later as `BFE - ground_elevation`.


In [ ]:
BFE_FIELDS = "ELEV,V_DATUM,LEN_UNIT,BFE_LN_ID,DFIRM_ID"

if BFE_PATH.exists():
    print(f"already have {BFE_PATH}; skipping fetch")
    bfe = gpd.read_parquet(BFE_PATH)
else:
    print("fetching S_BFE...")
    bfe = fetch_layer(BFE_LAYER_ID, out_fields=BFE_FIELDS)
    bfe.to_parquet(BFE_PATH, compression="zstd")
    print(f"wrote {BFE_PATH} ({len(bfe):,} features)")
bfe.head(3)


## Eyeball check: flood zones on a folium map

Color SFHA (1% annual chance — A/AE/AH/AO/V/VE) red, the 0.2% (shaded X) orange,
everything else grey. Sample SFHA only so the page stays responsive; that's the
visual we actually care about.


In [ ]:
SFHA = {"A", "AE", "AH", "AO", "AR", "A99", "V", "VE"}

def style_for(zone, subtype):
    if zone in SFHA:
        return {"color": "#b30000", "weight": 0.3, "fillColor": "#e34a33", "fillOpacity": 0.45}
    if zone == "X" and subtype and "0.2" in str(subtype):
        return {"color": "#b35900", "weight": 0.3, "fillColor": "#fdae6b", "fillOpacity": 0.35}
    return {"color": "#888888", "weight": 0.2, "fillColor": "#cccccc", "fillOpacity": 0.15}

sfha_only = zones[zones["FLD_ZONE"].isin(SFHA)]
sample = sfha_only.sample(min(len(sfha_only), 1500), random_state=0)

m = folium.Map(
    location=[(MIN_LAT + MAX_LAT) / 2, (MIN_LON + MAX_LON) / 2],
    zoom_start=10,
    tiles="cartodbpositron",
)
for _, row in sample.iterrows():
    style = style_for(row.get("FLD_ZONE"), row.get("ZONE_SUBTY"))
    folium.GeoJson(
        row.geometry.__geo_interface__,
        style_function=lambda _f, s=style: s,
        tooltip=f"{row.get('FLD_ZONE')} / {row.get('ZONE_SUBTY') or '-'}",
    ).add_to(m)
m


## Next

1. Pull a 3DEP DEM mosaic over the Harris County bbox; sample elevation at each
   building centroid (and vertices for buildings above a footprint threshold,
   taking the minimum to be conservative).
2. Spatial-join buildings to zones to assign `FLD_ZONE` and `STATIC_BFE` per
   building. For zones missing static BFE, interpolate from the nearest `S_BFE`
   line.
3. Compute `depth_100 = BFE - elevation`, clipped at 0; same for 500-yr using
   the X-shaded depth assumption.
4. Apply HAZUS depth-damage curves by archetype, integrate to EAD.
